# Personalized Career Path Recommendation with AI  
## Notebook 2: Recommendation Summary and Roadmap Generation

This notebook builds on the ranked career results produced in Notebook 1.

The goal here is to turn the scored output into more useful recommendation insights by:
- identifying the top career matches
- explaining why each role was recommended
- highlighting missing skills
- generating a simple upskilling roadmap

This makes the system more interpretable and more practical for end users.

In [1]:
import pandas as pd
import ast

## Load the ranked recommendation results

The previous notebook saved the ranked role output to an Excel file.

This file contains the career fit scores, matched skills, and missing skills for each role.

In [2]:
ranked_df = pd.read_excel("data/career_recommendation_results.xlsx")
ranked_df.head()

,role_title,domain,role_description,required_skills,preferred_skills,experience_level,education_level,interest_tags,required_skill_score,preferred_skill_score,interest_score,experience_score,education_score,fit_score,missing_required_skills,matched_required_skills
0,Data Analyst,Data & Analytics,"Analyses structured data to produce reports, d...","['SQL', 'Excel', 'Python', 'Data Visualization...","['Power BI', 'Tableau', 'Communication']",Entry,Bachelor/MSc,"['Analytics', 'Business', 'Problem Solving']",0.8,0.000000,0.666667,1.0,1,0.693333,['Excel'],"['SQL', 'Python', 'Data Visualization', 'Stati..."
1,Data Scientist,AI & Analytics,Builds predictive and statistical models to so...,"['Python', 'SQL', 'Statistics', 'Machine Learn...","['Experimentation', 'Communication', 'Deep Lea...",Entry/Junior,Bachelor/MSc,"['AI', 'Analytics', 'Modeling']",0.8,0.000000,0.666667,1.0,1,0.693333,['Machine Learning'],"['Python', 'SQL', 'Statistics', 'Data Visualiz..."
2,Product Analyst,Data & Analytics,Uses product and user data to evaluate feature...,"['SQL', 'Python', 'A/B Testing', 'Statistics',...","['Product Thinking', 'Communication', 'Dashboa...",Entry,Bachelor/MSc,"['Product', 'Experimentation', 'Analytics']",0.8,0.000000,0.333333,1.0,1,0.626667,['A/B Testing'],"['SQL', 'Python', 'Statistics', 'Data Visualiz..."
3,Decision Scientist,AI & Analytics,"Combines experimentation, modelling, and busin...","['Python', 'SQL', 'Statistics', 'Experimentati...","['Machine Learning', 'Data Visualization', 'Bu...",Junior,Bachelor/MSc,"['Decisions', 'Strategy', 'Analytics']",0.6,0.333333,0.333333,0.7,1,0.556667,"['Experimentation', 'Communication']","['Python', 'SQL', 'Statistics']"
4,Marketing Analyst,Marketing & Analytics,Uses campaign and customer data to measure per...,"['Excel', 'Data Visualization', 'Statistics', ...","['SQL', 'Python', 'A/B Testing']",Entry,Bachelor/MSc,"['Marketing', 'Analytics', 'Growth']",0.4,0.666667,0.333333,1.0,1,0.546667,"['Excel', 'Reporting', 'Communication']","['Data Visualization', 'Statistics']"


## Convert list-like text columns back into Python lists

When lists are saved to Excel, they are stored as plain text.

For example:

`['Python', 'SQL', 'Statistics']`

To work with them properly again, they must be converted back into Python lists.

In [3]:
list_like_cols = [
    "required_skills",
    "preferred_skills",
    "interest_tags",
    "missing_required_skills",
    "matched_required_skills"
]

def parse_list_column(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        return ast.literal_eval(value)
    except:
        return []

for col in list_like_cols:
    if col in ranked_df.columns:
        ranked_df[col] = ranked_df[col].apply(parse_list_column)

ranked_df.head()

,role_title,domain,role_description,required_skills,preferred_skills,experience_level,education_level,interest_tags,required_skill_score,preferred_skill_score,interest_score,experience_score,education_score,fit_score,missing_required_skills,matched_required_skills
0,Data Analyst,Data & Analytics,"Analyses structured data to produce reports, d...","[SQL, Excel, Python, Data Visualization, Stati...","[Power BI, Tableau, Communication]",Entry,Bachelor/MSc,"[Analytics, Business, Problem Solving]",0.8,0.000000,0.666667,1.0,1,0.693333,[Excel],"[SQL, Python, Data Visualization, Statistics]"
1,Data Scientist,AI & Analytics,Builds predictive and statistical models to so...,"[Python, SQL, Statistics, Machine Learning, Da...","[Experimentation, Communication, Deep Learning]",Entry/Junior,Bachelor/MSc,"[AI, Analytics, Modeling]",0.8,0.000000,0.666667,1.0,1,0.693333,[Machine Learning],"[Python, SQL, Statistics, Data Visualization]"
2,Product Analyst,Data & Analytics,Uses product and user data to evaluate feature...,"[SQL, Python, A/B Testing, Statistics, Data Vi...","[Product Thinking, Communication, Dashboarding]",Entry,Bachelor/MSc,"[Product, Experimentation, Analytics]",0.8,0.000000,0.333333,1.0,1,0.626667,[A/B Testing],"[SQL, Python, Statistics, Data Visualization]"
3,Decision Scientist,AI & Analytics,"Combines experimentation, modelling, and busin...","[Python, SQL, Statistics, Experimentation, Com...","[Machine Learning, Data Visualization, Busines...",Junior,Bachelor/MSc,"[Decisions, Strategy, Analytics]",0.6,0.333333,0.333333,0.7,1,0.556667,"[Experimentation, Communication]","[Python, SQL, Statistics]"
4,Marketing Analyst,Marketing & Analytics,Uses campaign and customer data to measure per...,"[Excel, Data Visualization, Statistics, Report...","[SQL, Python, A/B Testing]",Entry,Bachelor/MSc,"[Marketing, Analytics, Growth]",0.4,0.666667,0.333333,1.0,1,0.546667,"[Excel, Reporting, Communication]","[Data Visualization, Statistics]"


## Select the top career recommendations

The highest-ranked roles will be used as the main recommendations for the user.

This keeps the output focused and easier to interpret.

In [4]:
top_recommendations = ranked_df.head(3).copy()
top_recommendations[[
    "role_title",
    "domain",
    "fit_score",
    "matched_required_skills",
    "missing_required_skills"
]]

,role_title,domain,fit_score,matched_required_skills,missing_required_skills
0,Data Analyst,Data & Analytics,0.693333,"[SQL, Python, Data Visualization, Statistics]",[Excel]
1,Data Scientist,AI & Analytics,0.693333,"[Python, SQL, Statistics, Data Visualization]",[Machine Learning]
2,Product Analyst,Data & Analytics,0.626667,"[SQL, Python, Statistics, Data Visualization]",[A/B Testing]


## Generate a recommendation explanation

A recommendation system should not only rank roles, but also explain why they were selected.

This step creates a short explanation based on:
- matched skills
- career domain
- current fit score
- remaining skill gaps

In [5]:
def build_recommendation_explanation(row):
    matched = ", ".join(row["matched_required_skills"][:3]) if row["matched_required_skills"] else "some relevant skills"
    missing_count = len(row["missing_required_skills"])
    fit_pct = round(row["fit_score"] * 100, 1)

    explanation = (
        f"This role is a strong match because your current profile aligns with key skills such as {matched}. "
        f"It falls within the {row['domain']} domain and currently has a fit score of {fit_pct}%. "
        f"You still need to strengthen {missing_count} important skill(s) to become more competitive."
    )
    return explanation

top_recommendations["recommendation_explanation"] = top_recommendations.apply(
    build_recommendation_explanation, axis=1
)

top_recommendations[["role_title", "recommendation_explanation"]]

,role_title,recommendation_explanation
0,Data Analyst,This role is a strong match because your curre...
1,Data Scientist,This role is a strong match because your curre...
2,Product Analyst,This role is a strong match because your curre...


## Create a simple priority-based roadmap

Not all missing skills should be treated equally.

This step creates a simple learning roadmap by:
- prioritising required missing skills first
- limiting the focus to a manageable number
- turning skill gaps into action-oriented next steps

In [7]:
def build_learning_roadmap(missing_skills):
    if not missing_skills:
        return [
            "Strengthen current portfolio projects",
            "Practice interview questions",
            "Apply for relevant entry-level roles"
        ]

    roadmap = []
    for skill in missing_skills[:4]:
        roadmap.append(f"Learn or improve {skill}")
    roadmap.append("Build one portfolio project using these skills")
    roadmap.append("Update CV and LinkedIn to reflect new skills")
    return roadmap

top_recommendations["learning_roadmap"] = top_recommendations["missing_required_skills"].apply(
    build_learning_roadmap
)

top_recommendations[["role_title", "learning_roadmap"]]

,role_title,learning_roadmap
0,Data Analyst,"[Learn or improve Excel, Build one portfolio p..."
1,Data Scientist,"[Learn or improve Machine Learning, Build one ..."
2,Product Analyst,"[Learn or improve A/B Testing, Build one portf..."


## Create a more user-friendly recommendation summary

This step turns the top recommendations into a cleaner summary table that could later be shown in a dashboard or Streamlit app.

In [8]:
summary_df = top_recommendations[[
    "role_title",
    "domain",
    "fit_score",
    "matched_required_skills",
    "missing_required_skills",
    "recommendation_explanation",
    "learning_roadmap"
]].copy()

summary_df["fit_score"] = summary_df["fit_score"].round(3)

summary_df

,role_title,domain,fit_score,matched_required_skills,missing_required_skills,recommendation_explanation,learning_roadmap
0,Data Analyst,Data & Analytics,0.693,"[SQL, Python, Data Visualization, Statistics]",[Excel],This role is a strong match because your curre...,"[Learn or improve Excel, Build one portfolio p..."
1,Data Scientist,AI & Analytics,0.693,"[Python, SQL, Statistics, Data Visualization]",[Machine Learning],This role is a strong match because your curre...,"[Learn or improve Machine Learning, Build one ..."
2,Product Analyst,Data & Analytics,0.627,"[SQL, Python, Statistics, Data Visualization]",[A/B Testing],This role is a strong match because your curre...,"[Learn or improve A/B Testing, Build one portf..."


## Generate a top recommendation report

This step creates a readable recommendation report for the best-matching role.

This type of output can later be shown directly in the application interface.

In [9]:
best_role = top_recommendations.iloc[0]

print("Top Career Recommendation")
print("-" * 40)
print(f"Role: {best_role['role_title']}")
print(f"Domain: {best_role['domain']}")
print(f"Fit Score: {round(best_role['fit_score'] * 100, 1)}%")
print(f"Matched Skills: {', '.join(best_role['matched_required_skills'])}")
print(f"Missing Skills: {', '.join(best_role['missing_required_skills'])}")
print("\nWhy this role was recommended:")
print(best_role["recommendation_explanation"])
print("\nSuggested roadmap:")
for step_number, step in enumerate(best_role["learning_roadmap"], start=1):
    print(f"{step_number}. {step}")

Top Career Recommendation
----------------------------------------
Role: Data Analyst
Domain: Data & Analytics
Fit Score: 69.3%
Matched Skills: SQL, Python, Data Visualization, Statistics
Missing Skills: Excel

Why this role was recommended:
This role is a strong match because your current profile aligns with key skills such as SQL, Python, Data Visualization. It falls within the Data & Analytics domain and currently has a fit score of 69.3%. You still need to strengthen 1 important skill(s) to become more competitive.

Suggested roadmap:
1. Learn or improve Excel
2. Build one portfolio project using these skills
3. Update CV and LinkedIn to reflect new skills


## Save the recommendation summary

The summary output is saved for later use in the Streamlit app or final reporting notebook.

In [10]:
summary_df.to_excel("career_recommendation_summary.xlsx", index=False)
print("Saved as career_recommendation_summary.xlsx")

Saved as career_recommendation_summary.xlsx


## Summary

In this notebook, the ranked recommendation results were transformed into a more useful and explainable output.

The system now provides:
- top career recommendations
- recommendation explanations
- missing skill analysis
- a simple learning roadmap